# 🎓 Análisis Exploratorio: ¿Dónde estudian los niños de Barcelona? (cursos 2020/21 – 2024/25)

## Objetivo
**¿Dónde estudian los niños de Barcelona —en la escuela pública, concertada o privada— y depende del distrito en el que viven?**

1. ¿Qué proporción de alumnos estudia en cada tipo de centro, y cómo ha cambiado entre 2020/21 y 2024/25?
2. ¿Hay distritos donde predomina la escuela privada o concertada? ¿Cuáles?
3. ¿Influye el nivel de renta del distrito en el tipo de escuela que eligen las familias?

**Fuente:** Generalitat de Catalunya – *Alumnes matriculats per ensenyament i unitats dels centres docents* (API Socrata)

## 🛠️ Paso 0: Configuración inicial

In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 📥 Paso 1: Cargar los datos desde la API
Pedimos a la API solo los registros de Barcelona y de los últimos 5 cursos completos.

In [3]:
URL = "https://analisi.transparenciacatalunya.cat/resource/xvme-26kg.json"
CURSOS = ["2020/2021", "2021/2022", "2022/2023", "2023/2024", "2024/2025"]
LIMITE = 10000

lista_cursos = ",".join(f"'{c}'" for c in CURSOS)
filtro = f"nom_municipi='Barcelona' AND curs IN ({lista_cursos})"

paginas = []
offset = 0
while True:
    params = {
        "$where": filtro,
        "$limit": LIMITE,
        "$offset": offset,
        "$order": ":id",
    }
    respuesta = requests.get(URL, params=params)
    respuesta.raise_for_status()
    datos = respuesta.json()
    if not datos:
        break
    paginas.extend(datos)
    print(f"Descargadas {len(paginas)} filas...")
    offset += LIMITE

df = pd.DataFrame(paginas)
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

Descargadas 10000 filas...
Descargadas 20000 filas...
Descargadas 30000 filas...
Descargadas 40000 filas...
Descargadas 42611 filas...
Filas: 42611, Columnas: 35


### Guardar una copia local
Guardamos los datos en `data/raw/` para no tener que pedirlos a la API cada vez que ejecutemos el notebook.

In [4]:
df.to_csv("../data/raw/matriculas_bcn_raw.csv", index=False)
print("Guardado en data/raw/")

Guardado en data/raw/


## 👀 Paso 2: Primera exploración

### 2.1 Primeras filas

In [6]:
df.head()

,curs,any,codi_centre,denominaci_completa,codi_naturalesa,nom_naturalesa,codi_titularitat,nom_titularitat,codi_delegaci,nom_delegaci,...,nivell,modalitat,per_ode_matr_cula,matr_cules_total,matr_cules_dones,matr_cules_homes,unitats,matr_cula_concertada,tem_tica,grau
0,2022/2023,2022,08001595,Escola Baixeras,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,ANUAL,16,4,12,1,NaN,NaN,NaN
1,2022/2023,2022,08001595,Escola Baixeras,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,2,PRESENCIAL,ANUAL,18,7,11,1,NaN,NaN,NaN
2,2022/2023,2022,08001595,Escola Baixeras,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,3,PRESENCIAL,ANUAL,18,7,11,1,NaN,NaN,NaN
3,2022/2023,2022,08001595,Escola Baixeras,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,ANUAL,20,12,8,1,NaN,NaN,NaN
4,2022/2023,2022,08001595,Escola Baixeras,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,2,PRESENCIAL,ANUAL,21,10,11,1,NaN,NaN,NaN


### 2.2 Dimensiones del dataset

In [7]:
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

Filas: 42611, Columnas: 35


### 2.3 Información general: columnas, tipos y valores no nulos

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42611 entries, 0 to 42610
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   curs                      42611 non-null  str  
 1   any                       42611 non-null  str  
 2   codi_centre               42611 non-null  str  
 3   denominaci_completa       42611 non-null  str  
 4   codi_naturalesa           42611 non-null  str  
 5   nom_naturalesa            42611 non-null  str  
 6   codi_titularitat          42611 non-null  str  
 7   nom_titularitat           42611 non-null  str  
 8   codi_delegaci             42611 non-null  str  
 9   nom_delegaci              42611 non-null  str  
 10  codi_comarca              42611 non-null  str  
 11  nom_comarca               42611 non-null  str  
 12  codi_municipi_5           42611 non-null  str  
 13  codi_municip_6            42611 non-null  str  
 14  nom_municipi              42611 non-null  str  
 

### 2.4 Valores faltantes por columna

In [9]:
faltantes = df.isnull().sum()
porcentaje = (faltantes / len(df) * 100).round(1)
pd.DataFrame({"faltantes": faltantes, "%": porcentaje})[faltantes > 0].sort_values("%", ascending=False)

,faltantes,%
tem_tica,34952,82.0
grau,29592,69.4
matr_cula_concertada,22486,52.8
nivell,1536,3.6
codi_ensenyament,481,1.1
nom_ensenyament,482,1.1
coordenades_utm_x,30,0.1
coordenades_utm_y,30,0.1
coordenades_geo_x,30,0.1
coordenades_geo_y,30,0.1


### 2.5 Filas duplicadas

In [10]:
print(f"Filas duplicadas: {df.duplicated().sum()}")

Filas duplicadas: 26


In [12]:
df[df.duplicated(keep=False)].sort_values(["codi_centre", "curs"]).head(10)

,curs,any,codi_centre,denominaci_completa,codi_naturalesa,nom_naturalesa,codi_titularitat,nom_titularitat,codi_delegaci,nom_delegaci,...,nivell,modalitat,per_ode_matr_cula,matr_cules_total,matr_cules_dones,matr_cules_homes,unitats,matr_cula_concertada,tem_tica,grau
10062,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,26,18,8,0,NaN,NaN,IDILNA
10105,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,29,19,10,0,NaN,NaN,IDILNB
10125,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,24,15,9,0,NaN,NaN,IDILNB
10128,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,2,PRESENCIAL,2n QUADRIMESTRE,20,14,6,0,NaN,NaN,IDILNB
10146,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,8,3,5,0,NaN,NaN,IDILNI
10162,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,14,10,4,0,NaN,NaN,IDILNI
10180,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,26,18,8,0,NaN,NaN,IDILNA
10196,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,29,19,10,0,NaN,NaN,IDILNB
10200,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,1,PRESENCIAL,2n QUADRIMESTRE,24,15,9,0,NaN,NaN,IDILNB
10201,2020/2021,2020,08038570,EOI de Barcelona,1,Públic,01,Departament d'Ensenyament,0108,Consorci d'Educació de Barcelo,...,2,PRESENCIAL,2n QUADRIMESTRE,20,14,6,0,NaN,NaN,IDILNB


### 2.6 Valores únicos de los distritos

In [11]:
df["nom_dm"].value_counts()

nom_dm
Eixample                     6824
Sarrià - Sant Gervasi        5715
Sant Martí                   4938
Ciutat vella                 4622
Horta - Guinardó             4229
Nou Barris                   4188
Sants - Montjuïc             3961
Sant Andreu                  3145
Gràcia                       2610
Les Corts                    2314
DM VIII - Nou Barris           19
DM VII - Horta - Guinardó      19
DM II - Eixample                6
DM III - Sants - Montjuïc       6
DM IX - Sant Andreu             6
DM X - Sant Martí               5
DM I - Ciutat vella             4
Name: count, dtype: int64

### 2.7 ¿Los valores faltantes de `matr_cula_concertada` son de centros públicos?

In [13]:
pd.crosstab(df["nom_naturalesa"], df["matr_cula_concertada"].fillna("FALTANTE"))

matr_cula_concertada,FALTANTE,NO,SI
nom_naturalesa,,,
Privat,4096,6099,13046
Públic,18390,980,0


### 📝 Observaciones de la primera exploración
- **Tipos de datos:** todas las columnas llegan como texto (`str`), incluso las numéricas (`matr_cules_total`, `unitats`). Hay que convertirlas.
- **Distritos inconsistentes:** `nom_dm` tiene valores como "DM VIII - Nou Barris" además de "Nou Barris" (17 valores para 10 distritos). Además, "Ciutat vella" debería ser "Ciutat Vella" (importante para el merge con los datos de renta).
- **Concertada:** el 52% de `matr_cula_concertada` está vacío, pero el 82% de esos vacíos (18.390) son centros públicos, donde la pregunta no aplica (*faltante estructural*). El problema real son **4.096 registros privados** sin saber si son concertados o privados. Además, en centros públicos el mismo significado aparece como vacío o como "NO".
- **Duplicados:** 26 filas duplicadas, todas de dos Escoles Oficials d'Idiomes (enseñanza de adultos).
- **Coordenadas:** 30 filas sin coordenadas.
- **Columnas casi vacías:** `tem_tica` y `grau` solo están rellenas en algunos tipos de estudios.

### 🎯 Decisión de alcance
Nuestra pregunta es sobre **los niños**, así que nos quedamos con las etapas de 3 a 16 años:
- **Educació Infantil – segon cicle (P3-P5):** casi universal y se imparte en las escuelas. El primer ciclo (0-3) es voluntario y funciona como un sistema aparte (escoles bressol).
- **Educació Primària y ESO:** etapas obligatorias, todas las familias tienen que elegir centro.

Excluimos Batxillerat y FP (no obligatorios; el concierto suele cubrir solo las etapas obligatorias, lo que distorsionaría la comparación) y la enseñanza de adultos, idiomas y artísticas (fuera de la pregunta). Con este filtro, los 26 duplicados también quedan fuera.